In [1]:
from datasets import load_dataset
from transformers import (
    GPT2ForSequenceClassification, 
    GPT2Tokenizer, 
    Trainer, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    matthews_corrcoef,
    cohen_kappa_score,
    balanced_accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("COMPLETE EVALUATION METRICS GUIDE")
print("Using IMDB Dataset for Binary Classification")
print("="*80)

# ============================================================================
# STEP 1: Load Dataset
# ============================================================================

print("\n[1/6] Loading IMDB dataset...")

dataset = load_dataset("imdb")

# Use subset for faster training (you can increase this)
train_dataset_raw = dataset['train'].shuffle(seed=42).select(range(2000))
test_dataset_raw = dataset['test'].shuffle(seed=42).select(range(1000))

print(f"✓ Dataset loaded")
print(f"  Train: {len(train_dataset_raw)} samples")
print(f"  Test: {len(test_dataset_raw)} samples")

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-04 19:28:47.791448: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-04 19:28:53.422921: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-04 19:29:00.647536: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sli

COMPLETE EVALUATION METRICS GUIDE
Using IMDB Dataset for Binary Classification

[1/6] Loading IMDB dataset...
✓ Dataset loaded
  Train: 2000 samples
  Test: 1000 samples


In [2]:
# ============================================================================
# STEP 2: Prepare Model with LoRA
# ============================================================================

print("\n[2/6] Setting up model with LoRA...")

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

model = GPT2ForSequenceClassification.from_pretrained('gpt2', num_labels=2)
model.config.pad_token_id = tokenizer.pad_token_id

# Apply LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=["c_attn"],
)

model = get_peft_model(model, lora_config)
print("✓ LoRA applied")
model.print_trainable_parameters()


[2/6] Setting up model with LoRA...


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


✓ LoRA applied
trainable params: 296,448 || all params: 124,737,792 || trainable%: 0.2377


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [3]:
# ============================================================================
# STEP 3: Prepare Dataset
# ============================================================================

print("\n[3/6] Tokenizing dataset...")

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)

train_dataset = IMDBDataset(
    train_dataset_raw['text'],
    train_dataset_raw['label'],
    tokenizer
)

test_dataset = IMDBDataset(
    test_dataset_raw['text'],
    test_dataset_raw['label'],
    tokenizer
)

print("✓ Datasets prepared")


[3/6] Tokenizing dataset...
✓ Datasets prepared


In [4]:
# ============================================================================
# STEP 4: Define COMPREHENSIVE Metrics
# ============================================================================

print("\n[4/6] Defining comprehensive evaluation metrics...")

def compute_comprehensive_metrics(eval_pred):
    """
    Compute ALL important metrics for binary classification.
    This is what you should learn!
    """
    predictions, labels = eval_pred
    
    # Get predicted classes
    pred_classes = np.argmax(predictions, axis=1)
    
    # Get prediction probabilities
    pred_probs = torch.softmax(torch.tensor(predictions), dim=1)[:, 1].numpy()
    
    # ========================================================================
    # BASIC METRICS (Most Important)
    # ========================================================================
    
    # 1. ACCURACY: Overall correctness
    accuracy = accuracy_score(labels, pred_classes)
    
    # 2. PRECISION: Of all positive predictions, how many were correct?
    # "When I predict positive, how often am I right?"
    precision = precision_score(labels, pred_classes, zero_division=0)
    
    # 3. RECALL (Sensitivity): Of all actual positives, how many did I find?
    # "Of all the actual positives, how many did I catch?"
    recall = recall_score(labels, pred_classes, zero_division=0)
    
    # 4. F1-SCORE: Harmonic mean of precision and recall
    # "Balanced measure of precision and recall"
    f1 = f1_score(labels, pred_classes, zero_division=0)
    
    # ========================================================================
    # ADVANCED METRICS
    # ========================================================================
    
    # 5. SPECIFICITY: Of all actual negatives, how many did I correctly identify?
    # "True Negative Rate"
    tn, fp, fn, tp = confusion_matrix(labels, pred_classes).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # 6. BALANCED ACCURACY: Average of recall for each class
    # "Good for imbalanced datasets"
    balanced_acc = balanced_accuracy_score(labels, pred_classes)
    
    # 7. ROC-AUC: Area Under the ROC Curve
    # "Model's ability to distinguish between classes"
    try:
        roc_auc = roc_auc_score(labels, pred_probs)
    except:
        roc_auc = 0.5
    
    # 8. MATTHEWS CORRELATION COEFFICIENT: Overall quality for binary
    # "Good for imbalanced data, ranges from -1 to 1"
    mcc = matthews_corrcoef(labels, pred_classes)
    
    # 9. COHEN'S KAPPA: Agreement corrected for chance
    # "How much better than random guessing?"
    kappa = cohen_kappa_score(labels, pred_classes)
    
    # ========================================================================
    # CONFUSION MATRIX COMPONENTS
    # ========================================================================
    
    # True Positives: Correctly predicted positive
    # True Negatives: Correctly predicted negative
    # False Positives: Incorrectly predicted positive (Type I error)
    # False Negatives: Incorrectly predicted negative (Type II error)
    
    return {
        # Basic metrics
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        
        # Advanced metrics
        'specificity': specificity,
        'balanced_accuracy': balanced_acc,
        'roc_auc': roc_auc,
        'mcc': mcc,
        'kappa': kappa,
        
        # Confusion matrix components
        'true_positives': int(tp),
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn),
    }

print("✓ Comprehensive metrics defined")
print("\nMetrics that will be computed:")
print("  Basic: Accuracy, Precision, Recall, F1")
print("  Advanced: Specificity, Balanced Accuracy, ROC-AUC, MCC, Kappa")
print("  Components: TP, TN, FP, FN")


[4/6] Defining comprehensive evaluation metrics...
✓ Comprehensive metrics defined

Metrics that will be computed:
  Basic: Accuracy, Precision, Recall, F1
  Advanced: Specificity, Balanced Accuracy, ROC-AUC, MCC, Kappa
  Components: TP, TN, FP, FN


In [5]:
# ============================================================================
# STEP 5: Train Model
# ============================================================================

print("\n[5/6] Training model...")

training_args = TrainingArguments(
    output_dir='./results_metrics_demo',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_comprehensive_metrics,
)

print("Training with comprehensive metrics tracking...")
trainer.train()

print("\n✓ Training complete!")


[5/6] Training model...
Training with comprehensive metrics tracking...


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Specificity,Balanced Accuracy,Roc Auc,Mcc,Kappa,True Positives,True Negatives,False Positives,False Negatives
1,0.427100,0.366433,0.861000,0.918465,0.784836,0.846409,0.933594,0.859215,0.942711,0.728328,0.720888,383,478,34,105
2,0.315500,0.273641,0.878000,0.871951,0.879098,0.875510,0.876953,0.878026,0.959104,0.755930,0.755906,429,449,63,59
3,0.263800,0.289482,0.881000,0.847458,0.922131,0.883219,0.841797,0.881964,0.961614,0.765180,0.762354,450,431,81,38


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



✓ Training complete!


In [ ]:
# ============================================================================
# STEP 6: Comprehensive Evaluation and Visualization
# ============================================================================

print("\n[6/6] Running comprehensive evaluation...")

# Get predictions
predictions = trainer.predict(test_dataset)
pred_classes = np.argmax(predictions.predictions, axis=1)
true_labels = test_dataset_raw['label']

# Compute all metrics
results = compute_comprehensive_metrics((predictions.predictions, true_labels))

# ============================================================================
# DISPLAY RESULTS
# ============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE EVALUATION RESULTS")
print("="*80)

print("\n📊 BASIC METRICS (Most Important for Understanding):")
print("-" * 80)
print(f"  Accuracy:   {results['accuracy']:.4f}  ← Overall correctness")
print(f"  Precision:  {results['precision']:.4f}  ← Of predicted positives, % correct")
print(f"  Recall:     {results['recall']:.4f}  ← Of actual positives, % found")
print(f"  F1-Score:   {results['f1']:.4f}  ← Balanced precision & recall")

print("\n📈 ADVANCED METRICS:")
print("-" * 80)
print(f"  Specificity:       {results['specificity']:.4f}  ← Of actual negatives, % found")
print(f"  Balanced Accuracy: {results['balanced_accuracy']:.4f}  ← Good for imbalanced data")
print(f"  ROC-AUC:          {results['roc_auc']:.4f}  ← Separability (0.5=random, 1.0=perfect)")
print(f"  MCC:              {results['mcc']:.4f}  ← Quality (-1 to 1, 0=random)")
print(f"  Cohen's Kappa:    {results['kappa']:.4f}  ← Agreement beyond chance")

print("\n🔢 CONFUSION MATRIX BREAKDOWN:")
print("-" * 80)
print(f"  True Positives (TP):  {results['true_positives']:4d}  ← Correctly predicted positive")
print(f"  True Negatives (TN):  {results['true_negatives']:4d}  ← Correctly predicted negative")
print(f"  False Positives (FP): {results['false_positives']:4d}  ← Wrongly predicted positive (Type I error)")
print(f"  False Negatives (FN): {results['false_negatives']:4d}  ← Wrongly predicted negative (Type II error)")

# ============================================================================
# VISUAL CONFUSION MATRIX
# ============================================================================

print("\n" + "="*80)
print("CONFUSION MATRIX VISUALIZATION")
print("="*80)

# Create confusion matrix
cm = confusion_matrix(true_labels, pred_classes)

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix - IMDB Sentiment Classification')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('./confusion_matrix.png', dpi=150, bbox_inches='tight')
print("✓ Confusion matrix saved to './confusion_matrix.png'")

# ============================================================================
# DETAILED CLASSIFICATION REPORT
# ============================================================================

print("\n" + "="*80)
print("DETAILED CLASSIFICATION REPORT")
print("="*80)

report = classification_report(
    true_labels, 
    pred_classes, 
    target_names=['Negative', 'Positive'],
    digits=4
)
print(report)


[6/6] Running comprehensive evaluation...


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
